In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ==============================================================================
# 1. LOAD DATASET AND STANDARDISE HEADERS
# ==============================================================================
# Update this string to match your source CSV file name
df7 = pd.read_csv("Detailed_Polling_Data.csv")

# Standardise column spacing and strip whitespace to prevent key errors
df7.columns = df7.columns.str.replace(r'\s+', ' ', regex=True).str.strip()

# Wipe out any pre-existing calculated column copies to avoid value re-assignment errors
cols_to_clear = ['Margin_Percentage', 'Winner_Votes', 'Runner_Up_Votes', 'Margin_Of_Victory', 'Winner_Party', 'Cluster_ID']
df7 = df7.drop(columns=[c for c in cols_to_clear if c in df7.columns], errors='ignore')

# ==============================================================================
# 2. DEFINE DATASET CORE PARAMETERS
# ==============================================================================
core_parties = [
    'Dravida Munnetra Kazhagam',
    'All India Anna Dravida Munnetra Kazhagam', 
    'Tamilaga Vettri Kazhagam',
    'Naam Tamilar Katchi'
]
df7[core_parties] = df7[core_parties].fillna(0)

# Exact structural column definitions from your current dataset layout
station_col = 'Serial No. Of Polling Station'
building_col = 'Location and name of building in which Polling Station located'
area_col = 'Polling Areas'

# Isolate the 5 explicit independent candidate fields present in this specific layout
independent_candidate_cols = ['Independent', 'Independent.1', 'Independent.2', 'Independent.3', 'Independent.4']
existing_ind_cols = [c for c in independent_candidate_cols if c in df7.columns]
df7[existing_ind_cols] = df7[existing_ind_cols].fillna(0)

# Calculate total independent votes dynamically
df7['Total_Independent_Votes'] = df7[existing_ind_cols].sum(axis=1)

# ==============================================================================
# 3. CALCULATE NORMALIZED METRICS FOR THE MACHINE LEARNING MODEL
# ==============================================================================
df7['Total_Calculated_Votes'] = df7[core_parties].sum(axis=1) + df7['Total_Independent_Votes'] + df7['NOTA'].fillna(0)
df7 = df7[df7['Total_Calculated_Votes'] > 0].copy()

# Explicit Abbreviation Mapping to keep feature streams completely unique
party_abbreviations = {
    'Dravida Munnetra Kazhagam': 'DMK',
    'All India Anna Dravida Munnetra Kazhagam': 'AIADMK',
    'Tamilaga Vettri Kazhagam': 'TVK',
    'Naam Tamilar Katchi': 'NTK'
}

# Feature Engineering: Create normalized percentage shares (%)
share_cols = []
for party in core_parties:
    party_label = party_abbreviations[party]
    col_name = f'{party_label}_share_pct'
    
    if col_name in df7.columns:
        df7 = df7.drop(columns=[col_name])
        
    df7[col_name] = (df7[party] / df7['Total_Calculated_Votes']) * 100
    share_cols.append(col_name)

df7['independent_share_pct'] = (df7['Total_Independent_Votes'] / df7['Total_Calculated_Votes']) * 100

# Primary voting calculations (Winner, Runner-up, Margin)
df7['Winner_Votes'] = df7[core_parties].max(axis=1)
sorted_votes = np.sort(df7[core_parties].values, axis=1)
df7['Runner_Up_Votes'] = sorted_votes[:, -2]
df7['Margin_Of_Victory'] = df7['Winner_Votes'] - df7['Runner_Up_Votes']
df7['Winner_Party'] = df7[core_parties].idxmax(axis=1)
df7['Margin_Percentage'] = (df7['Margin_Of_Victory'] / df7['Total_Calculated_Votes']) * 100

# Set up clean target feature tracking array
feature_cols = share_cols + ['independent_share_pct', 'Margin_Percentage']
X = df7[feature_cols].copy().fillna(0)

# ==============================================================================
# 4. SCALE FEATURES AND RUN K-MEANS CLUSTERING
# ==============================================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
df7['Cluster_ID'] = kmeans.fit_predict(X_scaled)

# ==============================================================================
# 5. PRINT THE PROFILE SUMMARY BREAKDOWNS
# ==============================================================================
print("\n--- DATASET 7: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---")
print(df7.groupby('Cluster_ID')[feature_cols].mean().round(2))

print("\n--- DATASET 7: BOOTH COUNT PER CLUSTER ---")
print(df7['Cluster_ID'].value_counts())

# ==============================================================================
# 6. EXPORT STRATEGIC TARGET SHEETS FOR GROUND CAMPAIGN TEAMS
# ==============================================================================
# Dynamic placeholder tracking names. We can rename these once you print your numbers!
cluster_names = {
    0: "Cluster_0_Target", 
    1: "Cluster_1_Target", 
    2: "Cluster_2_Target", 
    3: "Cluster_3_Target"
}

for cluster_num in range(optimal_k):
    target_cols = [station_col, building_col, area_col, 'Winner_Party', 'Margin_Percentage']
    valid_target_cols = [c for c in target_cols if c in df7.columns]
    
    cluster_df = df7[df7['Cluster_ID'] == cluster_num][valid_target_cols]
    filename = f"Dataset_7_Cluster_{cluster_num}_{cluster_names[cluster_num]}.csv"
    cluster_df.to_csv(filename, index=False)

print("\nSuccess! Campaign target files generated cleanly for all 4 clusters.")


/Users/karthickkumarasamy/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (



--- DATASET 7: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---
            DMK_share_pct  AIADMK_share_pct  TVK_share_pct  NTK_share_pct  \
Cluster_ID                                                                  
0                   25.09             53.06          17.90           2.88   
1                   33.68             41.74          20.73           2.78   
2                   34.19             27.89          33.47           3.61   
3                   64.58             11.97          21.23           1.58   

            independent_share_pct  Margin_Percentage  
Cluster_ID                                            
0                            0.51              27.38  
1                            0.55               9.07  
2                            0.35               6.48  
3                            0.18              43.35  

--- DATASET 7: BOOTH COUNT PER CLUSTER ---
Cluster_ID
1    123
0    101
2     85
3     11
Name: count, dtype: int64

Success! Campaign target fil